## What Is Vector Store Memory?

### The core idea in one sentence
Instead of relying only on the **recent conversation window**, store older turns as **embeddings** and retrieve the most relevant memories when the user asks a new question.

---

### The mental model — a searchable memory library

Imagine every conversation turn becomes a page in a library. A normal buffer reads every page from the beginning. A sliding window reads only the last few pages. Vector Store Memory adds a librarian: when the user asks a question, the librarian finds the few older pages that are semantically related to that question.

This solves the biggest limitation of sliding windows: an important fact from turn 1 can still come back at turn 50, even if it is no longer in the recent context window.


In the earlier notebooks we saw three short-term memory patterns:

- Conversation Buffer Memory keeps everything, but token cost grows every turn.
- Sliding Window Memory caps token cost, but forgets old facts.
- Summary-based memory compresses old history, but may lose exact details.

Vector Store Memory takes a different path. It does not keep all old turns in the prompt. It stores them outside the prompt, converts them to embeddings, and retrieves only the most relevant ones when needed.

For FinCoach, this matters because users often share important facts early:

- salary
- expenses
- risk profile
- emergency fund
- financial goals
- dependents

Those facts may be needed much later, long after a sliding window has forgotten them.


### Key Concepts

**Embedding**: A list of numbers that represents the meaning of text. Similar meanings produce vectors that are close together.

**Vector store**: A database of embeddings plus the original text and metadata. In this notebook we build an in-memory vector store so the mechanism is visible.

**Similarity search**: Given a new query, embed the query and compare it against stored memories. The closest memories are retrieved.

**Top-k retrieval**: Retrieve only the best `k` memories. Larger `k` means better recall but higher prompt cost and more irrelevant context risk.

**Recent buffer + long-term retrieval**: Keep the last few turns verbatim for local coherence, and retrieve older relevant turns from the vector store.

**Retrieval miss**: The key failure mode. If the correct memory is not retrieved, the model cannot use it, even though it exists in storage.


### Mermaid source flowchart LR

    U[User message] --> Q[Embed query]
    Q --> S[Similarity search]
    S --> R[Top-k retrieved memories]

    subgraph Write Path
        T[Completed user + assistant turn] --> E[OpenAI embedding model]
        E --> V[(Vector memory store)]
    end

    V --> S
    B[Recent message buffer] --> P[Prompt builder]
    R --> P
    P --> L[OpenAI chat model]
    L --> A[Assistant reply]
    A --> T

### What this shows

- Every completed turn is stored after the assistant replies.
- The next user question is embedded and searched against stored turns.
- Only the top-k relevant older memories are injected into the prompt.
- Recent messages still go into the prompt verbatim.


In [ ]:
# Install required packages.
# 'openai'   — OpenAI SDK for chat and embedding API calls.
# 'tiktoken' — GPT-4o's exact tokeniser for token counting.
!pip install openai tiktoken --quiet


In [ ]:
# --- Standard library ---
import json                              # For serialising memory records to JSON.
import math                              # For cosine similarity and square roots.
import os                                # For reading environment variables.
import time                              # For pacing live API demos.
from collections import deque            # Recent message buffer with automatic eviction.
from datetime import datetime, timezone  # UTC timestamps, Python 3.12+ compatible.
from typing import List, Dict, Optional, Tuple  # Type hints for clarity and IDE support.
from dataclasses import dataclass, field, asdict  # Clean data models.

# --- Third-party ---
import openai    # OpenAI SDK.
import tiktoken  # Exact tokeniser for GPT-4o.


In [ ]:
# --- API Key Setup ---
# Option A: Colab Secrets (recommended)
try:
    from google.colab import userdata
    # google.colab is only available in Colab — this import fails locally.
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    # Fetch the key from Colab's encrypted secrets vault.
    print("Key loaded from Colab Secrets.")
except Exception:
    # Option B: Environment variable (for local Jupyter / VS Code).
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    print("Key loaded from environment variable.")

# Validate the key looks correct before we try to use it.
assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), \
    "API key missing or invalid. Add OPENAI_API_KEY to Colab Secrets or environment."

# Create the OpenAI client — the single object used for all API calls.
client = openai.OpenAI(api_key=OPENAI_API_KEY)

# Model constants.
MODEL = "gpt-4o"
# gpt-4o: GPT-4o — strong general-purpose model for the FinCoach responses.

EMBEDDING_MODEL = "text-embedding-3-small"
# text-embedding-3-small: cost-efficient OpenAI embedding model for semantic retrieval.

TOKENISER = tiktoken.get_encoding("o200k_base")
# o200k_base is the tokeniser family used by GPT-4o.
# Using the correct tokeniser means our token counts are close to the API's accounting.

print(f"Client ready. Chat model: {MODEL} | Embedding model: {EMBEDDING_MODEL}")


---
## Part 1 — The Message and Memory Data Models

`Message` is the same atomic unit as the earlier techniques.

`VectorMemoryRecord` is new. It stores:

- the original exchange text
- the OpenAI embedding vector
- metadata for filtering and debugging
- a timestamp for auditability


In [ ]:
@dataclass
class Message:
    """
    A single conversation message — role, content, and timestamp.
    Same data model as the earlier short-term memory techniques.
    """

    role: str
    # 'user' or 'assistant' — who sent this message.
    # The system prompt is handled separately, not stored here.

    content: str
    # The text of the message.

    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    # UTC timestamp — auto-set to now when the Message is created.

    def to_api_format(self) -> Dict[str, str]:
        """
        Strip the timestamp and return only what the OpenAI Chat Completions API accepts.
        """
        return {"role": self.role, "content": self.content}


@dataclass
class VectorMemoryRecord:
    """
    One long-term memory item stored in the vector store.
    Each record represents a completed user-assistant exchange.
    """

    memory_id: str
    # Stable ID for debugging and retrieval logs.

    user_id: str
    # User scope — prevents cross-user memory leakage in production.

    session_id: str
    # Session scope — useful for filtering and auditing.

    turn_number: int
    # The turn this memory came from.

    text: str
    # The original exchange text stored alongside the embedding.

    embedding: List[float]
    # Numeric vector returned by the OpenAI embedding model.

    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    # When this memory record was created.

print("Message and VectorMemoryRecord dataclasses defined.")


---
## Part 2 — Similarity Search Helpers

A vector store needs one mathematical operation: compare two vectors and decide how close they are.

We use **cosine similarity**:

- `1.0` means very similar direction
- `0.0` means unrelated direction
- negative values mean opposite direction

The OpenAI embedding API gives us the vectors. This helper only ranks them.


In [ ]:
def cosine_similarity(a: List[float], b: List[float]) -> float:
    """
    Compute cosine similarity between two embedding vectors.
    Higher score means the two pieces of text are semantically closer.
    """

    dot_product = sum(x * y for x, y in zip(a, b))
    # Dot product measures how much the vectors point in the same direction.

    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    # Vector length. We divide by this so longer vectors do not dominate.

    if norm_a == 0 or norm_b == 0:
        return 0.0
        # Defensive guard — OpenAI embeddings should not be zero vectors.

    return dot_product / (norm_a * norm_b)


def count_message_tokens(messages: List[Dict[str, str]]) -> int:
    """
    Approximate token count for a list of chat messages.
    We add a small per-message overhead for role/message formatting.
    """

    total = 0
    for message in messages:
        total += len(TOKENISER.encode(message.get("content", "")))
        total += 4
        # Small overhead for role and message framing.
    return total

print("Similarity and token helpers defined.")


---
## Part 3 — The VectorStoreMemory Class

### Why an in-memory vector store?

Production systems usually use Pinecone, Weaviate, Chroma, pgvector, Redis, Elasticsearch, or a managed memory platform.

For teaching, we build the vector store ourselves so every moving part is visible:

1. embed the completed exchange
2. store the vector and original text
3. embed the next user query
4. rank memories by cosine similarity
5. inject the best matches into the prompt

The retrieval idea is the same even when you swap this in-memory list for a production vector database.


In [ ]:
class VectorStoreMemory:
    """
    Stores old conversation turns as embedding vectors and retrieves
    semantically relevant memories for each new user message.

    Key difference from SlidingWindowMemory:
    - Window: remembers only what is recent.
    - Vector store: remembers what is relevant, even if it is old.
    """

    # ------------------------------------------------------------------
    # INITIALISATION
    # ------------------------------------------------------------------

    def __init__(
        self,
        session_id: str,             # Unique ID for this session.
        user_id: str,                # ID of the user — for multi-tenant isolation.
        system_prompt: str,          # Agent instructions — FinCoach's persona.
        top_k: int = 3,              # Number of memories to retrieve per query.
        recent_buffer_turns: int = 2 # Recent turns kept verbatim for local coherence.
    ):
        self.session_id = session_id
        # Identifies this session in logs and persistence.

        self.user_id = user_id
        # Scopes memory to one user. In production, always filter by user_id.

        self.system_prompt = system_prompt
        # FinCoach's persona and rules — prepended to every API call.

        self.top_k = top_k
        # How many retrieved memories to inject into the prompt.

        self.recent_buffer_turns = recent_buffer_turns
        # Number of recent user-assistant turns to keep verbatim.

        self._recent_messages: deque = deque(maxlen=recent_buffer_turns * 2)
        # Short-term context. This handles immediate conversational coherence.

        self._records: List[VectorMemoryRecord] = []
        # Long-term vector store. Each record contains text + embedding + metadata.

        self._total_turns = 0
        # Count of all completed user-assistant turns.

        self.retrieval_log: List[Dict] = []
        # Debug log showing which memories were retrieved for each query.

        self.created_at = datetime.now(timezone.utc).isoformat()
        # Session creation timestamp — useful for audit trail.

        print(f"[VectorStoreMemory] Initialised — session: {self.session_id}, "
              f"user: {self.user_id}, top_k: {self.top_k}, "
              f"recent buffer: {self.recent_buffer_turns} turns")

    # ------------------------------------------------------------------
    # EMBEDDING
    # ------------------------------------------------------------------

    def _embed(self, text: str) -> List[float]:
        """
        Convert text to an embedding vector using OpenAI.
        This is the write path for stored memories and the read path for queries.
        """

        response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=text,
        )
        # OpenAI returns a list of embedding objects. We sent one input string,
        # so we read response.data[0].embedding.

        return response.data[0].embedding

    # ------------------------------------------------------------------
    # WRITE PATH
    # ------------------------------------------------------------------

    def _store_exchange(self, user_message: str, assistant_reply: str) -> None:
        """
        Store a completed user-assistant exchange as one vector memory.
        We store after the assistant replies because the full turn is now known.
        """

        self._total_turns += 1
        # A completed turn = user message + assistant reply.

        exchange_text = (
            f"Turn {self._total_turns}\n"
            f"User: {user_message}\n"
            f"Assistant: {assistant_reply}"
        )
        # Store the original text too. Embeddings are for search;
        # the model still needs readable text in the prompt.

        embedding = self._embed(exchange_text)
        # Convert this exchange into a semantic vector.

        record = VectorMemoryRecord(
            memory_id=f"mem_{self._total_turns:04d}",
            user_id=self.user_id,
            session_id=self.session_id,
            turn_number=self._total_turns,
            text=exchange_text,
            embedding=embedding,
        )

        self._records.append(record)
        # Append to our in-memory vector store.

        self._recent_messages.append(Message(role="user", content=user_message))
        self._recent_messages.append(Message(role="assistant", content=assistant_reply))
        # Also keep the most recent turns verbatim.

        print(f"  [STORE] {record.memory_id} — embedded turn {self._total_turns} "
              f"({len(TOKENISER.encode(exchange_text))} text tokens)")

    # ------------------------------------------------------------------
    # READ PATH — RETRIEVAL
    # ------------------------------------------------------------------

    def retrieve(self, query: str, top_k: Optional[int] = None) -> List[Tuple[float, VectorMemoryRecord]]:
        """
        Retrieve the top-k most semantically similar stored memories.
        Returns a list of (similarity_score, memory_record) pairs.
        """

        if not self._records:
            return []
            # No memories yet — first turn has nothing to retrieve.

        k = top_k or self.top_k
        query_embedding = self._embed(query)
        # Embed the user's new question into the same vector space.

        scored_records = []
        for record in self._records:
            # In production, also filter by user_id and permissions here.
            if record.user_id != self.user_id:
                continue

            score = cosine_similarity(query_embedding, record.embedding)
            scored_records.append((score, record))

        scored_records.sort(key=lambda item: item[0], reverse=True)
        # Highest cosine similarity first.

        return scored_records[:k]

    def _format_retrieved_memories(
        self,
        retrieved: List[Tuple[float, VectorMemoryRecord]]
    ) -> str:
        """
        Convert retrieved memories into readable prompt context.
        """

        if not retrieved:
            return "No retrieved long-term memories yet."

        lines = []
        for rank, (score, record) in enumerate(retrieved, start=1):
            lines.append(
                f"Memory {rank} — {record.memory_id} — similarity {score:.3f}\n"
                f"{record.text}"
            )

        return "\n\n".join(lines)

    # ------------------------------------------------------------------
    # PROMPT BUILDING
    # ------------------------------------------------------------------

    def get_messages_for_api(
        self,
        current_user_message: str,
        retrieved: List[Tuple[float, VectorMemoryRecord]]
    ) -> List[Dict[str, str]]:
        """
        Build the API message list:
        1. system prompt
        2. retrieved long-term memories
        3. recent verbatim messages
        4. current user message
        """

        retrieved_context = self._format_retrieved_memories(retrieved)
        # Human-readable memories for the LLM.

        messages = [
            {"role": "system", "content": self.system_prompt},
            {
                "role": "system",
                "content": (
                    "Relevant long-term memories retrieved by semantic search. "
                    "Use them only if they help answer the user's current question.\n\n"
                    f"{retrieved_context}"
                ),
            },
        ]

        messages.extend(message.to_api_format() for message in self._recent_messages)
        # Recent messages preserve local flow and exact wording.

        messages.append({"role": "user", "content": current_user_message})
        # The current user message must be last.

        return messages

    # ------------------------------------------------------------------
    # CHAT LOOP
    # ------------------------------------------------------------------

    def chat(self, user_message: str, verbose: bool = True) -> str:
        """
        Retrieve relevant memories, call GPT-4o, store the completed turn,
        and return the assistant's reply.
        """

        retrieved = self.retrieve(user_message)
        # Retrieval happens BEFORE the current turn is stored.
        # The search is over previous memories only.

        messages = self.get_messages_for_api(user_message, retrieved)
        # Build prompt = system + retrieved memory + recent buffer + current message.

        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=1024,
            temperature=0.7,
            messages=messages,
        )

        assistant_reply = response.choices[0].message.content
        # Extract response text.

        self.retrieval_log.append({
            "turn": self._total_turns + 1,
            "query": user_message,
            "retrieved": [
                {
                    "memory_id": record.memory_id,
                    "turn_number": record.turn_number,
                    "score": round(score, 4),
                    "preview": record.text[:140],
                }
                for score, record in retrieved
            ],
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
        })
        # Store debugging info before adding the new exchange.

        self._store_exchange(user_message, assistant_reply)
        # Store the completed exchange for future turns.

        if verbose:
            retrieved_ids = [record.memory_id for _, record in retrieved]
            print(f"[Turn {self._total_turns}] API — prompt: {response.usage.prompt_tokens} tokens, "
                  f"completion: {response.usage.completion_tokens} tokens | "
                  f"retrieved: {retrieved_ids or 'none'} | "
                  f"stored memories: {len(self._records)}")

        return assistant_reply

    # ------------------------------------------------------------------
    # INSPECTION AND PERSISTENCE
    # ------------------------------------------------------------------

    def search(self, query: str, top_k: int = 5) -> None:
        """
        Debug search: print the top matching memories without calling the chat model.
        This is how you verify retrieval quality.
        """

        results = self.retrieve(query, top_k=top_k)
        print(f"\nSearch query: {query}")
        print("-" * 70)
        for rank, (score, record) in enumerate(results, start=1):
            print(f"#{rank} | {record.memory_id} | turn {record.turn_number} | score {score:.3f}")
            print(record.text[:350].replace("\n", " | "))
            print()

    def print_stats(self) -> None:
        """
        Print memory state for debugging and demos.
        """

        recent_turns = len(self._recent_messages) // 2
        print("\n=== VectorStoreMemory Stats ===")
        print(f"Session ID           : {self.session_id}")
        print(f"User ID              : {self.user_id}")
        print(f"Completed turns      : {self._total_turns}")
        print(f"Stored vector records: {len(self._records)}")
        print(f"Recent buffer turns  : {recent_turns}/{self.recent_buffer_turns}")
        print(f"Top-k retrieval      : {self.top_k}")
        print(f"Embedding model      : {EMBEDDING_MODEL}")

    def save(self, path: str) -> None:
        """
        Persist the vector memory to JSON.
        For real production, store embeddings in a vector database instead.
        """

        payload = {
            "session_id": self.session_id,
            "user_id": self.user_id,
            "system_prompt": self.system_prompt,
            "top_k": self.top_k,
            "recent_buffer_turns": self.recent_buffer_turns,
            "total_turns": self._total_turns,
            "created_at": self.created_at,
            "records": [asdict(record) for record in self._records],
            "recent_messages": [asdict(message) for message in self._recent_messages],
            "retrieval_log": self.retrieval_log,
        }

        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)

        print(f"Saved vector memory to {path}")

print("VectorStoreMemory class defined.")


---
## Part 4 — The FinCoach Agent with Vector Store Memory

The persona stays the same across techniques. Only the memory layer changes.

For Vector Store Memory, the chat flow is:

1. embed the current user message
2. retrieve relevant older turns
3. combine retrieved memories with the recent buffer
4. call GPT-4o
5. embed and store the completed turn


In [ ]:
# FinCoach's system prompt — same persona as the earlier notebooks.
# Consistent across all techniques — the agent's memory changes, not its identity.
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.

Your principles:
- Always personalise advice using information the user has shared in this conversation.
- Be specific with numbers when the user has provided their financial details.
- Flag when you are making assumptions due to missing information.
- Keep responses concise — 3 to 5 sentences unless the user asks for detail.
- Never provide specific buy/sell recommendations on individual stocks.
- Always recommend consulting a SEBI-registered advisor for major financial decisions.

Use retrieved long-term memories when they are relevant to the user's current question."""

print("FinCoach system prompt defined.")


---
## Part 5 — Demo: Semantic Recall After the Recent Window Has Moved

We use a tiny recent buffer of `2` turns so the old facts fall out quickly.

The salary is shared in Turn 1. Then we ask generic educational questions that do not require the salary. Finally, we ask for a plan that needs the old salary.

**Watch for:**

- `[STORE]` lines — each completed exchange becomes a vector memory
- `retrieved: [...]` lines — older memories coming back by semantic search
- Turn 6 — FinCoach can use the old salary even though it is no longer in the recent buffer


In [ ]:
vector_memory = VectorStoreMemory(
    session_id="session_vector_recall_demo",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    top_k=3,
    recent_buffer_turns=2,
)

demo_turns = [
    "Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.",
    # Turn 1: salary shared here.
    # This will soon fall out of the recent buffer.

    "My monthly expenses are about ₹60,000 for rent, food, and transport.",
    # Turn 2: expenses shared here.

    "I have an FD of ₹50,000 that matures in 3 months. I am risk-averse.",
    # Turn 3: risk profile and FD shared here.

    "What is a Systematic Investment Plan and how does it work?",
    # Turn 4: generic educational question.

    "What are the different types of mutual funds available in India?",
    # Turn 5: another generic question.

    "Given my salary and expenses, how much can I realistically save every month?",
    # Turn 6: requires old salary + expenses.
    # A sliding window with 2 recent turns would not reliably have those facts.
    # Vector memory should retrieve the old salary and expenses turns.
]

print("SEMANTIC RECALL DEMO — Vector Store Memory")
print("Strategy: old financial facts are retrieved when they become relevant again")
print("Expected: Turn 6 retrieves early salary/expense memories")
print("=" * 75)

for i, user_message in enumerate(demo_turns, start=1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {user_message}")
    response = vector_memory.chat(user_message=user_message, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

print("\n" + "=" * 75)
print("Demo complete.")
vector_memory.print_stats()


---
## Part 6 — Inspecting What Was Retrieved

Vector memory should never be a black box. During development, always inspect the retrieval log.

You want to know:

- Did the right memories come back?
- Were irrelevant memories injected?
- Did the model answer from retrieved memory or hallucinate?
- Is `top_k` too low or too high?


In [ ]:
print("=== Retrieval Log ===\n")

for entry in vector_memory.retrieval_log:
    print(f"Turn {entry['turn']}: {entry['query']}")
    print(f"Prompt tokens: {entry['prompt_tokens']} | Completion tokens: {entry['completion_tokens']}")

    if not entry["retrieved"]:
        print("  Retrieved: none — no stored memories yet")
    else:
        for item in entry["retrieved"]:
            print(f"  {item['memory_id']} | turn {item['turn_number']} | score {item['score']}")
            print(f"    {item['preview'].replace(chr(10), ' | ')}...")
    print("-" * 75)


In [ ]:
# Direct search is useful when tuning retrieval quality.
# No chat model call here — only embeddings + similarity search.

vector_memory.search(
    query="What salary and monthly expenses did the user mention?",
    top_k=5,
)

vector_memory.search(
    query="What is the user's risk profile and fixed deposit situation?",
    top_k=5,
)


---
## Part 7 — Cost Comparison: Buffer vs Sliding Window vs Vector Store

No API calls — pure token arithmetic.

Vector Store Memory has two costs:

1. **Embedding cost** when storing and searching memories.
2. **Prompt cost** for injecting the top-k retrieved memories.

The important pattern: the prompt does not grow with total conversation length. It grows with:

- system prompt
- recent buffer size
- top-k retrieved memories
- current user message


In [ ]:
def compare_buffer_window_vector(
    turns: List[str],
    window_size: int = 5,
    vector_recent_turns: int = 2,
    top_k: int = 3,
    avg_response_tokens: int = 80,
    avg_retrieved_memory_tokens: int = 120,
) -> None:
    """
    Compare prompt-token growth across three memory strategies.
    No API calls — pure arithmetic using tiktoken.
    """

    system_tokens = len(TOKENISER.encode(FINCOACH_SYSTEM_PROMPT))

    buffer_history_tokens = 0
    window_deque: deque = deque(maxlen=window_size * 2)
    vector_recent_deque: deque = deque(maxlen=vector_recent_turns * 2)

    print(f"System prompt      : {system_tokens} tokens")
    print(f"Sliding window     : {window_size} turns")
    print(f"Vector recent      : {vector_recent_turns} turns")
    print(f"Vector retrieval   : top_k={top_k}, approx {avg_retrieved_memory_tokens} tokens each")
    print()

    print(f"{'Turn':>4} | {'Buffer':>9} | {'Window':>9} | {'Vector':>9} | {'Vector retrieved':>16}")
    print("-" * 65)

    for i, user_msg in enumerate(turns, start=1):
        user_tokens = len(TOKENISER.encode(user_msg))

        # Buffer sends all historical tokens.
        buffer_history_tokens += user_tokens
        buffer_prompt = system_tokens + buffer_history_tokens
        buffer_history_tokens += avg_response_tokens

        # Sliding window sends only the last N turns.
        window_deque.append(user_tokens)
        window_deque.append(avg_response_tokens)
        window_prompt = system_tokens + sum(window_deque)

        # Vector memory sends recent turns + top-k retrieved memory snippets.
        vector_recent_deque.append(user_tokens)
        vector_recent_deque.append(avg_response_tokens)
        available_old_memories = max(0, i - vector_recent_turns)
        retrieved_count = min(top_k, available_old_memories)
        retrieved_tokens = retrieved_count * avg_retrieved_memory_tokens
        vector_prompt = system_tokens + sum(vector_recent_deque) + retrieved_tokens

        print(f"{i:>4} | {buffer_prompt:>9,} | {window_prompt:>9,} | "
              f"{vector_prompt:>9,} | {retrieved_count:>16}")

    print("-" * 65)
    print("Key insight:")
    print("  Buffer grows forever.")
    print("  Window stays cheap but forgets old facts.")
    print("  Vector memory stays bounded while selectively recalling old facts.")


simulated_turns = [
    "Hi, I'm Chiru. My monthly salary is ₹1,20,000.",
    "My monthly expenses are ₹60,000.",
    "I have an FD of ₹50,000 maturing in 3 months.",
    "I'm risk-averse and prefer stable returns.",
    "Should I invest in mutual funds or fixed deposits?",
    "What about SIPs? How much should I invest monthly?",
    "Is ₹10,000 per month in SIPs realistic given my expenses?",
    "Which fund categories suit a conservative investor?",
    "How long should I stay invested to see meaningful returns?",
    "Can you give me a complete monthly budget breakdown?",
    "What is the right emergency fund size for someone like me?",
    "Should I use a part of my FD to build the emergency fund?",
    "What tax-saving instruments should I look at under Section 80C?",
    "Is ELSS better than PPF for someone my age?",
    "Can you summarise the action plan we've discussed?",
]

compare_buffer_window_vector(simulated_turns)


---
## Part 8 — Tuning `top_k`: Recall vs Noise

`top_k` controls how many memories are injected into the prompt.

- Too low: the correct fact may not be included.
- Too high: irrelevant memories pollute the prompt and increase cost.

For many assistants, `top_k=3` to `top_k=5` is a good starting point. For highly technical or legal domains, tune it with evaluation data rather than intuition.


In [ ]:
def demonstrate_top_k_tradeoff(query: str, max_k: int = 6) -> None:
    """
    Show how retrieval context changes as top_k increases.
    This helps identify whether more memories improve recall or add noise.
    """

    print(f"Query: {query}")
    print("=" * 75)

    for k in range(1, max_k + 1):
        results = vector_memory.retrieve(query, top_k=k)
        prompt_tokens = 0
        for _, record in results:
            prompt_tokens += len(TOKENISER.encode(record.text))

        print(f"\ntop_k={k} | retrieved memories={len(results)} | memory text tokens={prompt_tokens}")
        for score, record in results:
            first_line = record.text.splitlines()[1] if len(record.text.splitlines()) > 1 else record.text[:80]
            print(f"  {record.memory_id} | score {score:.3f} | {first_line[:90]}")


demonstrate_top_k_tradeoff(
    query="Build a savings plan using my salary, expenses, FD, and risk profile.",
    max_k=5,
)


---
## Part 9 — The Retrieval Problem: Demonstrating What Can Go Wrong

Vector Store Memory is not magic. It can fail in two common ways:

1. **The memory exists but is not retrieved.** The embedding search ranks another memory higher.
2. **The memory is retrieved but ignored.** The prompt is too noisy or the model decides it is not relevant.

This is why production systems log retrievals and evaluate recall.


In [ ]:
def demonstrate_retrieval_visibility(
    query: str,
    expected_phrase: str,
    top_k_values: List[int]
) -> None:
    """
    Check whether an expected phrase appears in the retrieved memories.
    This is a simple recall test for memory retrieval.
    """

    print(f"Query           : {query}")
    print(f"Expected phrase : {expected_phrase}")
    print("-" * 75)

    for k in top_k_values:
        results = vector_memory.retrieve(query, top_k=k)
        retrieved_text = "\n".join(record.text for _, record in results).lower()
        phrase_visible = expected_phrase.lower() in retrieved_text

        status = "VISIBLE — model can use it" if phrase_visible else "MISSING — model cannot see it"
        print(f"top_k={k:<2} -> {status}")


demonstrate_retrieval_visibility(
    query="How much can Chiru save each month?",
    expected_phrase="₹1,20,000",
    top_k_values=[1, 2, 3, 5],
)


---
## Key Takeaways

**What you built:** A `VectorStoreMemory` class that stores completed turns as OpenAI embeddings, retrieves semantically relevant memories with cosine similarity, combines them with a recent message buffer, and sends the resulting context to GPT-4o.

---

### The progression so far

| | Technique 02 — Window | Technique 06 — Vector Store |
|---|---|---|
| Memory rule | Keep recent turns | Retrieve relevant turns |
| Token cost | Fixed after window fills | Bounded by recent buffer + top-k |
| Old fact recall | Poor after eviction | Strong if retrieval works |
| Exact transcript recall | Only recent turns | Retrieved stored turns |
| Main failure mode | Hard forgetting | Retrieval miss or noisy retrieval |
| Production role | Short-term memory | Long-term semantic memory |

---

### The three things to remember

1. **Vector memory separates storage from prompting.** You can store many past turns without sending all of them to the model every time.

2. **Retrieval quality is the product.** The model can only use memories that are retrieved and injected into the prompt. Always inspect retrieval logs.

3. **Use vector memory with a recent buffer.** Recent turns preserve conversational flow. Retrieved memories bring back older facts. The combination is much stronger than either one alone.

---

### What breaks next — and what Technique 07 fixes

Vector memory is good at semantic recall, but it stores text chunks rather than structured facts. If FinCoach needs a durable profile like:

- User: Chiru
- Salary: ₹1,20,000/month
- Risk profile: conservative
- Goal: emergency fund first

then Entity Memory is a cleaner layer. Technique 07 extracts named entities and stores structured facts that can be updated, queried, and reused across sessions.


### FAQ

Q: What is Vector Store Memory in agent memory?

A: Vector Store Memory stores conversation turns as embedding vectors and retrieves the most semantically relevant memories for each new user query. Instead of sending the full chat history, the agent sends only recent context plus top-k retrieved memories. This gives the agent long-range recall without unbounded prompt growth.

Q: How is Vector Store Memory different from Sliding Window Memory?

A: Sliding Window Memory remembers what is recent. Vector Store Memory remembers what is relevant. A sliding window forgets early facts once they fall out of the window. A vector store can retrieve those facts later if the new query is semantically related.

Q: What should I store in the vector database?

A: A strong default is one completed user-assistant exchange per memory record. Store the original text, embedding, user_id, session_id, turn number, timestamp, and any useful tags. For production, avoid storing sensitive data unless you have consent, encryption, retention policies, and deletion workflows.

Q: What is the best `top_k` value?

A: Start with `top_k=3` or `top_k=5`, then tune with evaluation data. A smaller `top_k` is cheaper and cleaner but can miss facts. A larger `top_k` improves recall but can add irrelevant context and increase cost. The right value depends on domain, chunk size, query style, and model behavior.

Q: Do I still need a recent conversation buffer?

A: Yes. Vector retrieval is not a replacement for short-term memory. Keep the last few turns verbatim so the model understands the immediate flow of the conversation. Use vector retrieval for older facts that are relevant but no longer recent.

Q: What production vector database should I use?

A: Common choices include pgvector, Pinecone, Weaviate, Chroma, Redis, Elasticsearch, and managed memory systems. The notebook uses an in-memory list so you can understand the mechanism. In production, swap the list for a real vector database with filtering by user_id, metadata, permissions, and retention policy.
